### What is Inference?
#### Definition
Inference is the process of using a trained model to make predictions on new, unseen data.

Diagram:
```
Training
    ↓
Save Model
    ↓
Load Model
    ↓
Inference
    ↓
Prediction
```

### Training vs Inference
| Training | Inference |
|-----------|------------|
| Learn Weights | Use Weights |
| Backpropagation | No Backpropagation |
| Optimizer Needed | No Optimizer |
| Slower | Faster |
| Requires Labels | Usually No Labels |
| Updates Parameters | Parameters Frozen |

### Big Picture
During Training
```
Input
 ↓
Model
 ↓
Prediction
 ↓
Loss
 ↓
Backpropagation
 ↓
Update Weights
```

During Inference
```
Input
 ↓
Model
 ↓
Prediction
```

That's it.

No learning happens.

### Typical Inference Pipeline
```
Load Model
     ↓
Receive Input
     ↓
Preprocess Input
     ↓
Model Prediction
     ↓
Postprocess Output
     ↓
Return Result
```

This pattern appears everywhere.

#### Step 1: Load Trained Model
```python
model = SimpleNN()

model.load_state_dict(
    torch.load(
        "model.pth"
    )
)
```

Diagram:
```
Saved Weights
      ↓
Loaded Model
```

#### Step 2: Switch to Evaluation Mode
```python
model.eval()
```

Why?

Disable:

- Dropout
- BatchNorm Updates

#### Step 3: Prepare Input
Example:
```python
x = torch.tensor(
    [[5.0, 3.0]]
)
```

Diagram:
```
User Input
      ↓
Tensor
```

#### Step 4: Disable Gradient Tracking
```python
with torch.no_grad():
```

Benefits:
```
Faster
Less Memory
```

because gradients are not needed.

#### Step 5: Make Prediction
```python
with torch.no_grad():

    prediction = model(x)
```

Diagram:
```
Input
 ↓
Model
 ↓
Prediction
```

### Complete Example
```python
import torch

model = SimpleNN()

model.load_state_dict(
    torch.load(
        "model.pth"
    )
)

model.eval()

x = torch.tensor(
    [[5.0, 3.0]]
)

with torch.no_grad():

    prediction = model(x)

print(prediction)
```

#### The Three Most Important Inference Lines
Almost every PyTorch inference script contains:

```python
model.eval()

with torch.no_grad():

    prediction = model(x)
```

Meaning:

##### 1. model.eval()
```
Disable Training Behaviors
```

##### 2. torch.no_grad()
```
Disable Gradient Tracking
```

##### 3. model(x)
```
Generate Prediction
```

### Preprocessing
Real-world data usually needs preprocessing before inference.

Diagram:
```
Raw Input
      ↓
Preprocessing
      ↓
Model
```

#### Why Preprocessing Matters
Suppose during training:
```
Scaling
Normalization
Tokenization
Encoding
```

were applied.

Inference must use:
```
Exactly The Same Preprocessing
```

Otherwise:
```
Bad Predictions
```

##### Example: Tabular Data
Training:
```python
X_scaled = scaler.fit_transform(X_train)
```

Inference:
```python
X_scaled = scaler.transform(X_new)
```

Important:
```
Use Existing Scaler

Do NOT Fit Again
```

### Example Pipeline
```
Raw Data
      ↓
Scaler / Tokenizer
      ↓
Tensor
      ↓
Model
      ↓
Prediction
```

### Postprocessing
After prediction:
```
Model Output
```

often needs conversion into a user-friendly result.


#### Example: Classification
Model output:
```
[0.1, 0.8, 0.1]
```

Meaning:
```
Class 0 → 10%

Class 1 → 80%

Class 2 → 10%
```

Final prediction:
```
Class 1
```

using:
```python
torch.argmax()
```

### Production Inference Pipeline
```
User Request
      ↓
Preprocessing
      ↓
Model
      ↓
Prediction
      ↓
Postprocessing
      ↓
Response
```

### Batch Inference
Instead of predicting one sample:
```
1 Input
```

we can predict:
```
100 Inputs
```

at once.

Example:
```python
predictions = model(
    batch_inputs
)
```

Diagram:
```
Input 1
Input 2
Input 3
Input 4
      ↓
Model
      ↓
Predictions
```

Benefits:
```
Higher Throughput
Better GPU Utilization
Lower Cost Per Prediction
```

### Single Sample vs Batch Inference
| Single | Batch |
|----------|----------|
| One request | Many requests |
| Lower throughput | Higher throughput |
| Simpler | More efficient |
| Real-time use | Bulk processing |


### Online vs Offline Inference
#### Online Inference
Real-time prediction.

Example:
```
User Sends Request
      ↓
Instant Prediction
```

Examples:
- Chatbots
- Search Engines
- Fraud Detection
- Recommendation APIs

#### Offline Inference
Predictions generated in batches.

Example:
```
Nightly Recommendation Job
```

Used for:
- Analytics
- Recommendations
- Reporting
- Forecasting

### Online vs Offline Comparison
| Online | Offline |
|----------|----------|
| Real-time | Batch |
| Low latency | High throughput |
| User waiting | User not waiting |
| API driven | Scheduled jobs |

### Deployment Architecture (High-Level)
Typical production flow:
```
Client
   ↓
API
   ↓
Model
   ↓
Prediction
   ↓
Client
```